#### Exercise 3 - Kraftwerkseinsatzplanung

VU - Energiemodelle und Analysen

Gruppe 10 - Hannah Bennoui, Peter Geyrhofer, Johannes Tanzer, Clemens Stadler

In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [26]:
# Parameterdefinition

power_coal = 600 #MW
power_gud = 400 #MW
power_gas = 300 #MW
power_pumps = 200 #MW
power_PV = 100 #MW
power_wind = 300 #MW
eta_coal = 0.41
eta_gud = 0.58
eta_gas = 0.4
eta_turbine = 0.9
eta_pump = 0.9
fuelcost_coal = 10 #€/MWh
fuelcost_gas = 45 #€/MWh
CO2_coal = 0.35 #%
CO2_ngas = 0.2 #%
CO2_price = 80 #€/t
capacity_pumps = 600 #MWh
H_coal = 8 # MWh/t
H_ngas = 14 # MWh/t
T = 24

In [27]:
# Import der .csv Datei und Umwandelung in pandas-dataframe

df_load = pd.read_csv('Last_PV_Wind.csv', sep=';')
df_load

,Uhrzeit,Last Winter [MW],Wind 300 MW,PV 100 MW Winter
0,01:00,720,160,0
1,02:00,680,134,0
2,03:00,640,126,0
3,04:00,620,115,0
4,05:00,620,92,0
5,06:00,660,112,0
6,07:00,940,103,2
7,08:00,1040,98,7
8,09:00,1140,139,15
9,10:00,1160,150,21


In [28]:
# Berechnung der Marginal Costs

mc_coal = fuelcost_coal/(eta_coal*H_coal)+(CO2_price*CO2_coal)/eta_coal
mc_gud = fuelcost_gas/(eta_gud*H_ngas)+(CO2_price*CO2_ngas)/eta_gud
mc_gas = fuelcost_gas/(eta_gas*H_ngas)+(CO2_price*CO2_ngas)/eta_gas
mc_pumps = 0
mc_PV = 0
mc_Wind = 0

print(mc_gas,mc_gud,mc_coal)

48.035714285714285 33.12807881773399 71.34146341463415


In [ ]:
# Minimierung der Kosten für den Kraftwerkseinsatz

kraftwerksnamen = ["GuD", "Gasturbine", "Kohle"]
n_units = 3  # Anzahl Kraftwerke
load = df_load['Last Winter [MW]']  # Beispiel-Lastkurve
co2_faktoren = [CO2_ngas, CO2_ngas, CO2_coal]
mc = [[mc_gud] * T,[mc_gas] * T,[mc_coal] * T] # Grenzkosten

# Maximale Leistung pro Einheit (z.B. 100 MW)
p_max = [power_gud, power_gas, power_coal]

# Modell erstellen
model = gp.Model("kraftwerkseinsatz")

# Entscheidungsvariablen: p[i][t] = Leistung Einheit i in Stunde t
p = {}
for i in range(n_units):
    for t in range(T):
        p[i, t] = model.addVar(lb=0, ub=p_max[i], name=f"p_{i}_{t}")

# Nachfragebedingung (Lastdeckung)
for t in range(T):
    model.addConstr(gp.quicksum(p[i, t] for i in range(n_units)) == load[t], name=f"load_{t}")

# Zielfunktion: Gesamtkosten minimieren
model.setObjective(
    gp.quicksum(mc[i][t] * p[i, t] for i in range(n_units) for t in range(T)),
    GRB.MINIMIZE
)

# Optimieren
model.optimize()

# Ergebnisse ausgeben
if model.status == GRB.OPTIMAL:
    for t in range(T):
        print(f"Stunde {t}:")
        for i in range(n_units):
            name = kraftwerksnamen[i]
            print(f"  {name}: {p[i, t].X:.2f} MW")

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i5-1235U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 24 rows, 72 columns and 72 nonzeros
Model fingerprint: 0xea8612df
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [3e+01, 7e+01]
  Bounds range     [3e+02, 6e+02]
  RHS range        [6e+02, 1e+03]
Presolve removed 19 rows and 57 columns
Presolve time: 0.01s
Presolved: 5 rows, 15 columns, 15 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.1473348e+06   1.525000e+02   0.000000e+00      0s
       5    1.1655221e+06   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.165522065e+06
Stunde 0:
  GuD: 400.00 MW
  Gasturbine: 300.00 MW
  Kohle: 20.00 MW
Stunde 1:
  GuD: 400.00 MW
  Gasturbine: 280.00 M